# Decision Analysis and Data Validation Using Benford's Law
This notebook analyzes decision data and validates it using Benford's Law, applying statistical tests to detect anomalies.

## Background on Benford's Law
Benford's Law predicts the distribution of digits in naturally occurring datasets, useful for data integrity analysis.

## Key Metrics and Variables
Key terms in this notebook:
- **Cramér’s V**: Indicates association strength between variables.
- **p-value**: Measures statistical significance.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from benfordslaw import benfordslaw
from scipy.stats import norm, chi2_contingency, ks_2samp
from pathlib import Path

import os
import numpy as np
import pandas as pd
from benfordslaw import benfordslaw
from scipy.stats import norm

In [2]:
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
results_dir = '.temp/experiment_results'

In [3]:
def format_summary(df):
    '''Formats summary tables for better readability.'''
    return df.round(3)

## Data Overview
The data consists of decisions from different experiments, analyzed across budgets and episodes.

In [4]:

def read_parquet_files_into_dataframe(folder_path: str) -> pd.DataFrame:
    """
    List all parquet files in the specified folder and read them into a single DataFrame.
    
    :param folder_path: The directory containing parquet files.
    :return: A concatenated DataFrame containing data from all parquet files.
    """
    # List all .parquet files in the folder
    parquet_files = [f for f in Path(folder_path).glob("*.parquet")]
    print(parquet_files)

    # Check if there are parquet files in the folder
    if not parquet_files:
        print(f"No parquet files found in {folder_path}")
        return pd.DataFrame()

    # Read all parquet files into a list of DataFrames
    dataframes = []
    for file in parquet_files:
        print(f"Reading {file}...")
        df = pd.read_parquet(file)
        dataframes.append(df)

    # Concatenate all DataFrames into a single DataFrame
    combined_dataframe = pd.concat(dataframes, axis=0, ignore_index=True)
    combined_dataframe["steps"] = pd.to_numeric(combined_dataframe["steps"])
    combined_dataframe["rewards"] = pd.to_numeric(combined_dataframe["rewards"] )
    combined_dataframe["episodes"] = pd.to_numeric(combined_dataframe["episodes"] )

    return combined_dataframe

# Example usage:
random_folder_path = "..\\.temp\\data\\experiment_results\\random\\results"
dqn_folder_path = "..\\.temp\\data\\experiment_results\\dqn\\insurenv\\"


random_df = read_parquet_files_into_dataframe(random_folder_path).assign(experiment='random')
dqn_df = read_parquet_files_into_dataframe(dqn_folder_path).assign(experiment='dqn')


data = pd.concat(
    [random_df, 
    dqn_df], 
    axis = 0).query("initial_budget in (10_000_000,20_000_000, 30_000_000) ")


[WindowsPath('../.temp/data/experiment_results/random/results/results_10000000.parquet'), WindowsPath('../.temp/data/experiment_results/random/results/results_20000000.parquet'), WindowsPath('../.temp/data/experiment_results/random/results/results_30000000.parquet')]
Reading ..\.temp\data\experiment_results\random\results\results_10000000.parquet...
Reading ..\.temp\data\experiment_results\random\results\results_20000000.parquet...
Reading ..\.temp\data\experiment_results\random\results\results_30000000.parquet...
[WindowsPath('../.temp/data/experiment_results/dqn/insurenv/results_10000000.parquet'), WindowsPath('../.temp/data/experiment_results/dqn/insurenv/results_20000000.parquet'), WindowsPath('../.temp/data/experiment_results/dqn/insurenv/results_30000000.parquet')]
Reading ..\.temp\data\experiment_results\dqn\insurenv\results_10000000.parquet...
Reading ..\.temp\data\experiment_results\dqn\insurenv\results_20000000.parquet...
Reading ..\.temp\data\experiment_results\dqn\insurenv\

## Initial Data Exploration
We explore the structure and quality of the loaded data.

In [5]:
data.pivot_table(index = 'experiment', 
                 columns= ['initial_budget'], values= 'decision', aggfunc= ['sum','mean', 'std', 'count']).T

experiment                      dqn       random
      initial_budget                            
sum   10000000         18904.000000   447.000000
      20000000         39148.000000   683.000000
      30000000         40825.000000   837.000000
mean  10000000             0.062660     0.475532
      20000000             0.056426     0.503687
      30000000             0.054492     0.491774
std   10000000             0.242351     0.499667
      20000000             0.230744     0.500171
      30000000             0.226986     0.500079
count 10000000        301691.000000   940.000000
      20000000        693788.000000  1356.000000
      30000000        749190.000000  1702.000000

In [18]:
import plotly.express as px

fig = px.box(
    data.groupby(['experiment', 'initial_budget', 'decision', 'episodes'])['cum_rewards'].max().reset_index(),
    y='cum_rewards',
    color='decision',
    facet_col='initial_budget',
    facet_row='experiment',
    height=1200,
    title='Cumulative Rewards by Decision Across Experiments and Initial Budgets',
    labels={
        "cum_rewards": "Cumulative Rewards",
        "episodes": "Episodes",
        "initial_budget": "Initial Budget",
        "experiment": "Experiment"
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
    #boxmode="overlay"
)

# Update layout to free y-axis for each subplot
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)

# Additional layout adjustments for clarity
#fig.update_layout(
#    title_x=0.5,
#    margin=dict(l=50, r=50, t=100, b=50),
#    showlegend=True,
#    boxgap=0.3,
#    plot_bgcolor="rgba(0, 0, 0, 0)"
#)
#
## Custom hover template for detailed tooltips
#fig.update_traces(
#    hovertemplate="<br>Decision: %{color}<br>Cumulative Reward: %{y}<br>Experiment: %{row}<br>Initial Budget: %{col}"
#)
#
# Show gridlines only for y-axis
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")
fig.update_xaxes(showgrid=False)

fig.show()


In [7]:
data.pivot_table(index = 'experiment', 
                 columns= ['initial_budget'], values= 'cum_rewards', aggfunc= ['sum','mean', 'std', 'count']).T.round()

experiment                   dqn  random
      initial_budget                    
sum   10000000         6248351.0  1603.0
      20000000        23745712.0  2793.0
      30000000        34670453.0  4035.0
mean  10000000              21.0     2.0
      20000000              34.0     2.0
      30000000              46.0     2.0
std   10000000              29.0     2.0
      20000000              40.0     3.0
      30000000              44.0     3.0
count 10000000          301691.0   940.0
      20000000          693788.0  1356.0
      30000000          749190.0  1702.0

In [28]:
fig = px.line(data.groupby([
    'experiment', 
    'initial_budget', 
    'episodes'
])['cum_rewards'].max().reset_index(), 
        x = 'episodes', 
        y = 'cum_rewards', 
        color = 'initial_budget', 
        facet_row='experiment')

fig.update_yaxes(matches=None)

In [24]:
fig = px.line(data.groupby([
    'experiment', 
    'initial_budget', 
    'episodes'
])['steps'].max().reset_index(), 
        x = 'episodes', 
        y = 'steps', 
        color = 'initial_budget', 
        facet_row='experiment')

fig.update_yaxes(matches=None)

In [25]:
fig = px.line(data.groupby([
    'experiment', 
    'initial_budget', 
    'episodes'
])['decision'].mean().reset_index(), 
        x = 'episodes', 
        y = 'decision', 
        color = 'initial_budget', 
        facet_row='experiment')

fig.update_yaxes(matches=None)

In [45]:
import plotly.express as px

# Generate the line chart with independent Y-axes
fig = px.line(
    data.groupby(
        ['experiment', 'initial_budget', 'episodes', 'decision']
    )['valor_indenizacao'].median().reset_index(),
    x='episodes',
    y='valor_indenizacao',
    color='initial_budget',
    facet_row='experiment',
    facet_col='decision', 
    height=1000,
    width=1000,
    title="Median Indemnity Value by Initial Budget Across Experiments",
    labels={
        "valor_indenizacao": "Median Indemnity Value",
        "episodes": "Episodes",
        "initial_budget": "Initial Budget",
        "experiment": "Experiment"
    },
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Free each facet's Y-axis
fig.update_yaxes(matches=None)

# Additional layout customization
fig.update_layout(
    title_x=0.5,
    margin=dict(l=50, r=50, t=80, b=50),
    legend_title_text="Initial Budget",
    plot_bgcolor="rgba(0, 0, 0, 0)"  # Transparent background for cleaner look
)

# Custom hover template to improve readability
fig.update_traces(
    hovertemplate="<br>Episode: %{x}<br>Median Indemnity: %{y}<br>Initial Budget: %{legendgroup}"
)

# Fine-tune grid and line appearance for readability
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

# Display the final chart
fig.show()


### Initial Data Insights
Observations on data completeness and structure.

In [ ]:
px.box(
    data, 
    x= 'episodes', 
    y = 'cum_rewards', 
    color = 'decision', 
    facet_col='experiment', 
    facet_row = 'initial_budget', 
    height=2000
)

## Benford’s Law Analysis Setup
Setting up parameters and functions for Benford's Law analysis.

In [46]:
pos_values = [1, 2, 3]
alpha = 0.05

In [47]:
def benford_analysis(data, digit_pos, alpha):
    bl = benfordslaw(method='chi', alpha=alpha, pos=digit_pos)
    results = bl.fit(data)
    return results

In [62]:
# Assuming your dataset is loaded into a DataFrame called 'data'
# The dataset includes columns: experiment, rewards, wc, woc,
# valor_indenizacao, initial_budget, decision, episodes

# Define the positions to analyze (first, second, and third digits)
pos_values = [1, 2, 3]

# Group the data by experiment, decision, initial_budget, and episodes
groups = data.groupby(['experiment', 'decision', 'initial_budget', 'episodes'])

# Prepare a list to store summary results
summary_results = []

# Confidence level for intervals (e.g., 95%)
confidence_level = 0.95
z_score = norm.ppf(1 - (1 - confidence_level) / 2)

# Iterate over each group and apply Benford's Law analysis
for (experiment, decision, initial_budget, episodes), group in groups:
    valor_indenizacao = group['valor_indenizacao'].dropna().values
    
    if len(valor_indenizacao) == 0:
        print(
            f"No indemnity values available for Experiment: {experiment}, "
            f"Decision: {decision}, Initial Budget: {initial_budget}, Episodes: {episodes}"
        )
        continue
    
    for pos in pos_values:
        # Initialize the Benford's Law analyzer for this position
        bl = benfordslaw(method='chi', alpha=0.10, pos=pos)
        
        # Apply Benford's Law to the valor_indenizacao values
        results = bl.fit(valor_indenizacao)
        
        # Calculate Cramér's V
        chi_squared = results['t']  # Chi-squared statistic
        n = len(valor_indenizacao)
        k = len(results['percentage_emp'])  # Number of categories
        cramers_v = np.sqrt(chi_squared / (n * (k - 1)))
        
        # Calculate confidence intervals for observed proportions
        observed_props = results['percentage_emp'] / 100
        confidence_intervals = []
        for p in observed_props:
            se = np.sqrt(p * (1 - p) / n)
            ci_lower = p - z_score * se
            ci_upper = p + z_score * se
            # Ensure the confidence intervals are within [0, 1]
            ci_lower = np.maximum(ci_lower, 0)
            ci_upper = np.minimum(ci_upper, 1)
            confidence_intervals.append((ci_lower, ci_upper))
        
        # Append results to the summary list
        summary_results.append({
            'experiment': experiment,
            'decision': decision,
            'initial_budget': initial_budget,
            'episodes': episodes,
            'digit_position': pos,
            'statistic': chi_squared,
            'p_value': results['P'],
            'is_significant': results['P_significant'],
            'cramers_v': cramers_v,
            'observed_proportions': observed_props,
            'confidence_intervals': confidence_intervals,
        })

# Create a summary DataFrame
summary_df = pd.DataFrame(summary_results)

# Save the summary DataFrame to the results directory
summary_filename = os.path.join(results_dir, 'benford_summary_results.csv')
summary_df.to_csv(summary_filename, index=False)
print(f"Summary results saved to: {summary_filename}")

[benfordslaw] >Analyzing digit position: [1]
[benfordslaw] >[P_ensemble] Anomaly detected! P=0.0136084, Tstat=12.5651
[benfordslaw] >Analyzing digit position: [2]
[benfordslaw] >[P_ensemble] Anomaly detected! P=0.0394277, Tstat=10.0601
[benfordslaw] >Analyzing digit position: [3]
[benfordslaw] >[P_ensemble] Anomaly detected! P=0.0463716, Tstat=9.66986
[benfordslaw] >Analyzing digit position: [1]
[benfordslaw] >[P_ensemble] No anomaly detected. P=0.204843, Tstat=5.9247
[benfordslaw] >Analyzing digit position: [2]
[benfordslaw] >[P_ensemble] No anomaly detected. P=0.196742, Tstat=6.03239
[benfordslaw] >Analyzing digit position: [3]
[benfordslaw] >[P_ensemble] No anomaly detected. P=0.104666, Tstat=7.66461
[benfordslaw] >Analyzing digit position: [1]
[benfordslaw] >[P_ensemble] No anomaly detected. P=0.5493, Tstat=3.05116
[benfordslaw] >Analyzing digit position: [2]
[benfordslaw] >[P_ensemble] No anomaly detected. P=0.506436, Tstat=3.31578
[benfordslaw] >Analyzing digit position: [3]
[ben

In [63]:
import pandas as pd
from scipy.stats import ks_2samp

# Assuming summary_df is already loaded as a pandas DataFrame

# Get unique agents, initial budgets, and digit positions
agents = summary_df['experiment'].unique()
initial_budgets = summary_df['initial_budget'].unique()
digit_positions = summary_df['digit_position'].unique()

# Iterate over each agent
for agent in agents:
    # Iterate over each initial budget
    for budget in initial_budgets:
        # Iterate over each digit position
        for digit_pos in digit_positions:
            # Filter data for the current agent, initial budget, and digit position
            subset_df = summary_df[
                (summary_df['experiment'] == agent) &
                (summary_df['initial_budget'] == budget) &
                (summary_df['digit_position'] == digit_pos)
            ]
            
            # Check if subset_df is not empty
            if subset_df.empty:
                continue  # Skip if no data is available for this combination

            # Separate the p-values based on the decision (approved or rejected)
            approved_pvalues = subset_df[subset_df['decision'] == 1]['p_value']
            rejected_pvalues = subset_df[subset_df['decision'] == 0]['p_value']
            
            # Ensure there are enough data points to perform the KS test
            if len(approved_pvalues) >= 2 and len(rejected_pvalues) >= 2:
                # Perform the KS test
                ks_statistic, ks_pvalue = ks_2samp(approved_pvalues, rejected_pvalues)

                # Output the results
                print(f"Agent: {agent}, Initial Budget: {budget}, Digit Position: {digit_pos}")
                print(f"KS Statistic: {ks_statistic:.4f}, P-value: {ks_pvalue:.4f}")
                print("---------------------------------------------")
            else:
                print(f"Insufficient data for Agent: {agent}, Initial Budget: {budget}, Digit Position: {digit_pos}")
                print("---------------------------------------------")


Agent: dqn, Initial Budget: 10000000, Digit Position: 1
KS Statistic: 0.5862, P-value: 0.0000
---------------------------------------------
Agent: dqn, Initial Budget: 10000000, Digit Position: 2
KS Statistic: 0.3793, P-value: 0.0000
---------------------------------------------
Agent: dqn, Initial Budget: 10000000, Digit Position: 3
KS Statistic: 0.2586, P-value: 0.0008
---------------------------------------------
Agent: dqn, Initial Budget: 20000000, Digit Position: 1
KS Statistic: 0.4476, P-value: 0.0000
---------------------------------------------
Agent: dqn, Initial Budget: 20000000, Digit Position: 2
KS Statistic: 0.3007, P-value: 0.0000
---------------------------------------------
Agent: dqn, Initial Budget: 20000000, Digit Position: 3
KS Statistic: 0.3776, P-value: 0.0000
---------------------------------------------
Agent: dqn, Initial Budget: 30000000, Digit Position: 1
KS Statistic: 0.4793, P-value: 0.0000
---------------------------------------------
Agent: dqn, Initial 

In [64]:
summary_df = pd.DataFrame(summary_results)
summary_df.sample(10)

,experiment,decision,initial_budget,episodes,digit_position,statistic,p_value,is_significant,cramers_v,observed_proportions,confidence_intervals
2616,random,0.0,10000000,112,1,14.478455,0.005915,True,1.345291,"[[0.01, 1.0], [0.02, 0.0], [0.03, 0.0], [0.04,...","[([0.0, 1.0], [0.20501395417987875, 1.0]), ([0..."
3576,random,1.0,10000000,6,1,9.652631,0.046704,True,0.634187,"[[0.01, 0.0], [0.02, 0.33333333333333326], [0....","[([0.0, 0.0], [0.12259135894148633, 0.0]), ([0..."
2041,dqn,1.0,30000000,42,2,3.922299,0.416624,False,0.043156,"[[0.0, 0.06837606837606838], [0.01, 0.13675213...","[([0.0, 0.03603811407023384], [0.0, 0.10071402..."
4443,random,1.0,30000000,11,1,5.065630,0.280629,False,0.324860,"[[0.01, 0.33333333333333326], [0.02, 0.0], [0....","[([0.0, 0.0], [0.0896141134105336, 0.710528578..."
2073,dqn,1.0,30000000,53,1,4.368365,0.358442,False,0.043096,"[[0.01, 0.2687074829931973], [0.02, 0.16326530...","[([0.0, 0.21803643681803164], [0.0213734447729..."
2966,random,0.0,20000000,86,3,6.449508,0.168000,False,0.319958,"[[0.0, 0.0], [0.01, 0.14285714285714285], [0.0...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.083..."
2980,random,0.0,20000000,91,2,7.208346,0.125279,False,0.400232,"[[0.0, 0.0], [0.01, 0.4], [0.02, 0.0], [0.03, ...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.097..."
4300,random,1.0,20000000,105,2,19.024257,0.000777,True,1.453893,"[[0.0, 0.0], [0.01, 0.0], [0.02, 0.0], [0.03, ...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.205..."
4604,random,1.0,30000000,64,3,6.456949,0.167523,False,0.320143,"[[0.0, 0.0], [0.01, 0.0], [0.02, 0.28571428571...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.083..."
4566,random,1.0,30000000,52,1,2.036775,0.728995,False,0.152135,"[[0.01, 0.18181818181818182], [0.02, 0.0909090...","[([0.0, 0.0], [0.06879891953620162, 0.40974494..."


In [65]:
import pandas as pd
from scipy.stats import ks_2samp

# Assuming summary_df is already loaded as a pandas DataFrame

# Initialize an empty list to store the results
results = []

# Get unique agents, initial budgets, and digit positions
agents = summary_df['experiment'].unique()
initial_budgets = summary_df['initial_budget'].unique()
digit_positions = summary_df['digit_position'].unique()

# Iterate over each agent
for agent in agents:
    # Iterate over each initial budget
    for budget in initial_budgets:
        # Iterate over each digit position
        for digit_pos in digit_positions:
            # Filter data for the current agent, initial budget, and digit position
            subset_df = summary_df[
                (summary_df['experiment'] == agent) &
                (summary_df['initial_budget'] == budget) &
                (summary_df['digit_position'] == digit_pos)
            ]
            
            # Check if subset_df is not empty
            if subset_df.empty:
                continue  # Skip if no data is available for this combination

            # Separate the p-values based on the decision (approved or rejected)
            approved_pvalues = subset_df[subset_df['decision'] == 1]['p_value']
            rejected_pvalues = subset_df[subset_df['decision'] == 0]['p_value']
            
            # Ensure there are enough data points to perform the KS test
            if len(approved_pvalues) >= 2 and len(rejected_pvalues) >= 2:
                # Perform the KS test
                ks_statistic, ks_pvalue = ks_2samp(approved_pvalues, rejected_pvalues)
                
                # Append the results to the list
                results.append({
                    'Agent': agent,
                    'Initial Budget': budget,
                    'Digit Position': digit_pos,
                    'KS Statistic': ks_statistic,
                    'P-Value': ks_pvalue
                })
            else:
                # Append a result indicating insufficient data
                results.append({
                    'Agent': agent,
                    'Initial Budget': budget,
                    'Digit Position': digit_pos,
                    'KS Statistic': None,
                    'P-Value': None,
                    'Note': 'Insufficient data'
                })

# Convert the results list into a DataFrame
results_df = pd.DataFrame(results)

# Remove entries with insufficient data
results_df = results_df[results_df['KS Statistic'].notnull()]

# Format the KS Statistic and P-Value columns
results_df['KS Statistic'] = results_df['KS Statistic'].apply(lambda x: f"{x:.4f}")
results_df['P-Value'] = results_df['P-Value'].apply(lambda x: f"{x:.4f}")

# Sort the results for better readability
results_df.sort_values(by=['Agent', 'Initial Budget', 'Digit Position'], inplace=True)

# Reset index
results_df.reset_index(drop=True, inplace=True)

# Display the summary DataFrame
print(results_df)

# Optionally, you can display the results in a tabular format using pandas
from tabulate import tabulate

# Prepare the DataFrame for display
display_df = results_df[['Agent', 'Initial Budget', 'Digit Position', 'KS Statistic', 'P-Value']]

# Display the table
print(tabulate(display_df, headers='keys', tablefmt='psql', showindex=False))


     Agent  Initial Budget  Digit Position KS Statistic P-Value
0      dqn        10000000               1       0.5862  0.0000
1      dqn        10000000               2       0.3793  0.0000
2      dqn        10000000               3       0.2586  0.0008
3      dqn        20000000               1       0.4476  0.0000
4      dqn        20000000               2       0.3007  0.0000
5      dqn        20000000               3       0.3776  0.0000
6      dqn        30000000               1       0.4793  0.0000
7      dqn        30000000               2       0.4215  0.0000
8      dqn        30000000               3       0.3802  0.0000
9   random        10000000               1       0.1127  0.3291
10  random        10000000               2       0.0845  0.6929
11  random        10000000               3       0.0915  0.5930
12  random        20000000               1       0.1127  0.3291
13  random        20000000               2       0.1831  0.0170
14  random        20000000              

## Benford Results Summary
Description of key columns in `summary_df`.

In [66]:
summary_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4836 entries, 0 to 4835
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   experiment            4836 non-null   object 
 1   decision              4836 non-null   float64
 2   initial_budget        4836 non-null   int64  
 3   episodes              4836 non-null   int64  
 4   digit_position        4836 non-null   int64  
 5   statistic             4836 non-null   float64
 6   p_value               4836 non-null   float64
 7   is_significant        4836 non-null   bool   
 8   cramers_v             4836 non-null   float64
 9   observed_proportions  4836 non-null   object 
 10  confidence_intervals  4836 non-null   object 
dtypes: bool(1), float64(4), int64(3), object(3)
memory usage: 382.7+ KB


In [88]:
px.box(summary_df, 
       y = 'p_value', facet_row = 'experiment', 
       facet_col='initial_budget', 
       color='decision')

In [93]:
import plotly.express as px

# Create the box plot with enhancements
fig = px.box(
    summary_df,
    x='digit_position',
    y='p_value',
    color='decision',
    facet_row='experiment',
    facet_col='initial_budget',
    title="P-Value Distribution by Digit Position Across Experiments and Initial Budgets",
    labels={
        "p_value": "P-Value",
        "digit_position": "Digit Position",
        "experiment": "Experiment",
        "initial_budget": "Initial Budget",
        "decision": "Decision"
    },
    color_discrete_sequence=px.colors.qualitative.Bold,
    #boxmode="overlay",  # Overlay boxes for direct comparison within each facet
    height=1000,
    width=1200
)

# Free each facet's Y-axis for better visibility across different scales
fig.update_yaxes(matches=None)

# Layout customization for improved readability
fig.update_layout(
    title_x=0.5,
    margin=dict(l=50, r=50, t=100, b=50),
    showlegend=True,
    boxgap=0.3,
    legend_title_text="Decision",
    plot_bgcolor="rgba(0, 0, 0, 0)"
)

# Customize hover template to make tooltips clear and informative
fig.update_traces(
    hovertemplate="<br>Digit Position: %{x}<br>P-Value: %{y}<br>Decision: %{color}"
)

# Show gridlines only for y-axis and adjust for clarity
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")
fig.update_xaxes(showgrid=False)

# Display the enhanced chart
fig.show()


In [94]:
import plotly.express as px

# Create the box plot with independent Y-axes for each facet
fig = px.box(
    summary_df.query("episodes > 110"),
    x='experiment',
    y='p_value',
    color='decision',
    facet_row='digit_position',
    facet_col='initial_budget',
    height=1000,
    title="P-Value Distribution by Experiment and Initial Budget Across Digit Positions",
    labels={
        "p_value": "P-Value",
        "experiment": "Experiment",
        "digit_position": "Digit Position",
        "initial_budget": "Initial Budget",
        "decision": "Decision"
    },
    color_discrete_sequence=px.colors.qualitative.Vivid,
    #boxmode="overlay"  # Overlay boxes for better comparison within each facet
)

# Free each facet's Y-axis
fig.update_yaxes(matches=None)

# Additional layout customization for clarity
fig.update_layout(
    title_x=0.5,
    margin=dict(l=50, r=50, t=80, b=50),
    showlegend=True,
    boxgap=0.3,
    legend_title_text="Decision",
    plot_bgcolor="rgba(0, 0, 0, 0)"
)

# Custom hover template for detailed tooltips
fig.update_traces(
    hovertemplate="<br>Experiment: %{x}<br>P-Value: %{y}<br>Decision: %{color}"
)

# Fine-tune grid and background settings for readability
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")
fig.update_xaxes(showgrid=False)

# Display the improved chart
fig.show()


In [95]:
summary_df.query("experiment=='dqn'")

,experiment,decision,initial_budget,episodes,digit_position,statistic,p_value,is_significant,cramers_v,observed_proportions,confidence_intervals
0,dqn,0.0,10000000,1,1,12.565101,0.013608,True,0.723565,"[[0.01, 0.0], [0.02, 0.33333333333333326], [0....","[([0.0, 0.0], [0.12259135894148633, 0.0]), ([0..."
1,dqn,0.0,10000000,1,2,10.060079,0.039428,True,0.610406,"[[0.0, 0.0], [0.01, 0.0], [0.02, 0.0], [0.03, ...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.122..."
2,dqn,0.0,10000000,1,3,9.669860,0.046372,True,0.598450,"[[0.0, 0.0], [0.01, 0.33333333333333326], [0.0...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.122..."
3,dqn,0.0,10000000,2,1,5.924702,0.204843,False,0.304259,"[[0.01, 0.25], [0.02, 0.25], [0.03, 0.375], [0...","[([0.0, 0.0], [0.07894784471329745, 0.55005697..."
4,dqn,0.0,10000000,2,2,6.032394,0.196742,False,0.289453,"[[0.0, 0.0], [0.01, 0.0], [0.02, 0.125], [0.03...","[([0.0, 0.0], [0.0, 0.0]), ([0.0, 0.0], [0.078..."
...,...,...,...,...,...,...,...,...,...,...,...
2275,dqn,1.0,30000000,120,2,0.832186,0.934082,False,0.011973,"[[0.0, 0.10077519379844961], [0.01, 0.12093023...","[([0.0, 0.07754358990757083], [0.0, 0.12400679..."
2276,dqn,1.0,30000000,120,3,6.108908,0.191161,False,0.032440,"[[0.0, 0.10232558139534884], [0.01, 0.10852713...","[([0.0, 0.0789361443664358], [0.0, 0.125715018..."
2277,dqn,1.0,30000000,121,1,1.350447,0.852760,False,0.016091,"[[0.01, 0.30368098159509205], [0.02, 0.1610429...","[([0.002362664124756864, 0.2683840284056962], ..."
2278,dqn,1.0,30000000,121,2,1.014103,0.907650,False,0.013146,"[[0.0, 0.09969325153374232], [0.01, 0.11963190...","[([0.0, 0.07669724484117149], [0.0, 0.12268925..."


In [98]:
import plotly.express as px

# Create the line chart with independent Y-axes for each facet
fig = px.line(
    summary_df.query("initial_budget == 10_000_000"),
    x='episodes',
    y='cramers_v',
    color='decision',
    facet_row='experiment',
    facet_col='digit_position',
    height=1000,
    title="Cramer's V Over Episodes by Decision Across Experiments and Digit Positions (Initial Budget: 10M)",
    labels={
        "cramers_v": "Cramer's V",
        "episodes": "Episodes",
        "experiment": "Experiment",
        "digit_position": "Digit Position",
        "decision": "Decision"
    },
    color_discrete_sequence=px.colors.qualitative.Plotly  # Consistent color scheme
)

# Free each facet's Y-axis
fig.update_yaxes(matches=None)

# Additional layout customization for better visual clarity
fig.update_layout(
    title_x=0.5,
    margin=dict(l=50, r=50, t=100, b=50),
    legend_title_text="Decision",
    plot_bgcolor="rgba(0, 0, 0, 0)"  # Transparent background for a cleaner look
)

# Custom hover template to improve readability
fig.update_traces(
    hovertemplate="<br>Episode: %{x}<br>P-Value: %{y}<br>Decision: %{legendgroup}"
)

# Grid and background settings to keep the plot clean
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

# Display the final chart
fig.show()


In [ ]:
import plotly.express as px

# Create the line chart with independent Y-axes for each facet
fig = px.line(
    summary_df.query("initial_budget == 20_000_000"),
    x='episodes',
    y='cramers_v',
    color='decision',
    facet_row='experiment',
    facet_col='digit_position',
    height=1000,
    title="Cramer's V Over Episodes by Decision Across Experiments and Digit Positions (Initial Budget: 20M)",
    labels={
        "cramers_v": "Cramer's V",
        "episodes": "Episodes",
        "experiment": "Experiment",
        "digit_position": "Digit Position",
        "decision": "Decision"
    },
    color_discrete_sequence=px.colors.qualitative.Plotly  # Consistent color scheme
)

# Free each facet's Y-axis
fig.update_yaxes(matches=None)

# Additional layout customization for better visual clarity
fig.update_layout(
    title_x=0.5,
    margin=dict(l=50, r=50, t=100, b=50),
    legend_title_text="Decision",
    plot_bgcolor="rgba(0, 0, 0, 0)"  # Transparent background for a cleaner look
)

# Custom hover template to improve readability
fig.update_traces(
    hovertemplate="<br>Episode: %{x}<br>P-Value: %{y}<br>Decision: %{legendgroup}"
)

# Grid and background settings to keep the plot clean
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

# Display the final chart
fig.show()

In [102]:
import plotly.express as px

# Create the line chart with independent Y-axes for each facet
fig = px.line(
    summary_df.query("initial_budget == 30_000_000"),
    x='episodes',
    y='cramers_v',
    color='decision',
    facet_row='experiment',
    facet_col='digit_position',
    height=1000,
    title="Cramer's V Over Episodes by Decision Across Experiments and Digit Positions (Initial Budget: 30M)",
    labels={
        "cramers_v": "Cramer's V",
        "episodes": "Episodes",
        "experiment": "Experiment",
        "digit_position": "Digit Position",
        "decision": "Decision"
    },
    color_discrete_sequence=px.colors.qualitative.Plotly  # Consistent color scheme
)

# Free each facet's Y-axis
fig.update_yaxes(matches=None)

# Additional layout customization for better visual clarity
fig.update_layout(
    title_x=0.5,
    margin=dict(l=50, r=50, t=100, b=50),
    legend_title_text="Decision",
    plot_bgcolor="rgba(0, 0, 0, 0)"  # Transparent background for a cleaner look
)

# Custom hover template to improve readability
fig.update_traces(
    hovertemplate="<br>Episode: %{x}<br>P-Value: %{y}<br>Decision: %{legendgroup}"
)

# Grid and background settings to keep the plot clean
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgrey")

# Display the final chart
fig.show()

## Benford results and final inference
Overview of planned visualizations for Benford's Law results.

In [107]:
summary_df.groupby(["experiment", "initial_budget", "digit_position", "decision"]).describe().reset_index().T.reset_index().to_csv("final_ananlysis.csv", index = False)

In [109]:
summary_df.groupby(["experiment", "initial_budget", "digit_position", "decision"]).describe().reset_index().T.reset_index()

,level_0,level_1,0,1,2,3,4,5,6,7,...,26,27,28,29,30,31,32,33,34,35
0,experiment,,dqn,dqn,dqn,dqn,dqn,dqn,dqn,dqn,...,random,random,random,random,random,random,random,random,random,random
1,initial_budget,,10000000,10000000,10000000,10000000,10000000,10000000,20000000,20000000,...,20000000,20000000,20000000,20000000,30000000,30000000,30000000,30000000,30000000,30000000
2,digit_position,,1,1,2,2,3,3,1,1,...,2,2,3,3,1,1,2,2,3,3
3,decision,,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
4,episodes,count,116.0,116.0,116.0,116.0,116.0,116.0,143.0,143.0,...,142.0,142.0,142.0,142.0,142.0,142.0,142.0,142.0,142.0,142.0
5,episodes,mean,58.5,58.5,58.5,58.5,58.5,58.5,72.0,72.0,...,70.5,70.5,70.5,70.5,70.5,70.5,70.5,70.5,70.5,70.5
6,episodes,std,33.630343,33.630343,33.630343,33.630343,33.630343,33.630343,41.42463,41.42463,...,41.135953,41.135953,41.135953,41.135953,41.135953,41.135953,41.135953,41.135953,41.135953,41.135953
7,episodes,min,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,episodes,25%,29.75,29.75,29.75,29.75,29.75,29.75,36.5,36.5,...,35.25,35.25,35.25,35.25,35.25,35.25,35.25,35.25,35.25,35.25
9,episodes,50%,58.5,58.5,58.5,58.5,58.5,58.5,72.0,72.0,...,70.5,70.5,70.5,70.5,70.5,70.5,70.5,70.5,70.5,70.5


In [111]:
# Check the structure and column names of the DataFrame to understand its format
print(summary_df.columns)

# Display the first few rows to understand its content
print(summary_df.head())


Index(['experiment', 'decision', 'initial_budget', 'episodes',
       'digit_position', 'statistic', 'p_value', 'is_significant', 'cramers_v',
       'observed_proportions', 'confidence_intervals'],
      dtype='object')
  experiment  decision  initial_budget  episodes  digit_position  statistic  \
0        dqn       0.0        10000000         1               1  12.565101   
1        dqn       0.0        10000000         1               2  10.060079   
2        dqn       0.0        10000000         1               3   9.669860   
3        dqn       0.0        10000000         2               1   5.924702   
4        dqn       0.0        10000000         2               2   6.032394   

    p_value  is_significant  cramers_v  \
0  0.013608            True   0.723565   
1  0.039428            True   0.610406   
2  0.046372            True   0.598450   
3  0.204843           False   0.304259   
4  0.196742           False   0.289453   

                                observed_proportion

In [112]:
import pandas as pd
from scipy.stats import ttest_ind

# Assuming `summary_df` is already loaded as per the provided structure

# Filter for DQN and Random agent data
dqn_data = summary_df[summary_df['experiment'] == 'dqn']
random_data = summary_df[summary_df['experiment'] == 'random']

# Extract Benford distance statistic for both agents if it's available as 'statistic' or another metric
dqn_statistics = dqn_data['statistic'].astype(float)
random_statistics = random_data['statistic'].astype(float)

# Conduct Welch's t-test for unequal variances
t_stat, p_value = ttest_ind(dqn_statistics, random_statistics, equal_var=False)

# Print the results
print(f"T-statistic: {t_stat}, P-value: {p_value}")

# Interpret the result
if p_value < 0.05:
    print("Conclusion: The DQN agent achieves a significantly lower Benford distance than the random agent.")
else:
    print("Conclusion: No significant difference in Benford distances between the DQN and random agents.")


T-statistic: -47.66768242842811, P-value: 0.0
Conclusion: The DQN agent achieves a significantly lower Benford distance than the random agent.


In [113]:
# Assuming the t_stat and p_value are from the two-tailed test
if t_stat < 0:  # Check if the direction supports DQN being lower
    one_tailed_p_value = p_value / 2  # Convert to one-tailed

    print(f"One-tailed P-value: {one_tailed_p_value}")
    if one_tailed_p_value < 0.05:
        print("Conclusion: The DQN agent achieves a significantly lower Benford distance than the random agent.")
    else:
        print("Conclusion: No significant evidence that the DQN agent's Benford distance is lower.")
else:
    print("Since the t-statistic is not negative, we cannot conclude that DQN is lower.")


One-tailed P-value: 0.0
Conclusion: The DQN agent achieves a significantly lower Benford distance than the random agent.


In [114]:
from scipy.stats import mannwhitneyu

# Assuming dqn_statistics and random_statistics contain the chi-squared values for each agent
u_stat, p_value = mannwhitneyu(dqn_statistics, random_statistics, alternative='less')

# Output the results
print(f"Mann-Whitney U-statistic: {u_stat}, P-value: {p_value}")

# Interpretation
if p_value < 0.05:
    print("Conclusion: The DQN agent achieves significantly lower chi-squared values than the random agent.")
else:
    print("Conclusion: No significant evidence that the DQN agent's chi-squared values are lower.")


Mann-Whitney U-statistic: 950369.5, P-value: 0.0
Conclusion: The DQN agent achieves significantly lower chi-squared values than the random agent.
